# SOTA Comparison — Stage 3 (ours) vs GFPGAN v1.4 vs CodeFormer

**Mục tiêu**: so sánh fair phương pháp đề xuất với 2 baseline SOTA trên cùng 200 ảnh CelebA-HQ test, cùng Vietnam Degradation Pipeline (medium + hard).

**Datasets cần attach** (Add Data):
1. Code project v6+ (`mrtaintou/vn-old-portrait-restore-v7` hoặc tương đương, đã có `sota_comparison.py`)
2. Pretrained: `mrtaintou/vn-pretrained-model-gan-v1` (GFPGANv1.4.pth + RealESRGAN_x4plus.pth)
3. Stage 3 checkpoint: `mrtaintou/vn-stage3-final-completed`
4. CelebA-HQ test: `badasstechie/celebahq-resized-256x256`

**Internet**: bật ON để clone CodeFormer repo + download codeformer.pth.

**GPU**: T4 (15 GB) hoặc P100 đều đủ.

**Thời gian**: ~25-35 phút cho 200 ảnh × 2 levels × 3 methods.

## 1. Verify datasets

In [ ]:
from pathlib import Path
print('Input datasets:')
for p in sorted(Path('/kaggle/input').iterdir()):
    n = sum(1 for _ in p.rglob('*') if _.is_file())
    print(f'  {p.name:<50} ({n} files)')

## 2. Setup code project

In [ ]:
import os, sys, shutil
from pathlib import Path

DST = '/kaggle/working/vn-old-portrait-restoration'
SRC = None
for p in Path('/kaggle/input').iterdir():
    hits = list(p.rglob('src/pipeline/restore_pipeline.py'))
    if hits:
        SRC = str(hits[0].parent.parent.parent)
        break
assert SRC, 'Code project (chứa restore_pipeline.py) không tìm thấy'

if os.path.exists(DST):
    shutil.rmtree(DST)
shutil.copytree(SRC, DST)
os.chdir(DST)
if DST not in sys.path:
    sys.path.insert(0, DST)

required = [
    'src/pipeline/restore_pipeline.py',
    'scripts/evaluate_pipeline.py',
    'src/degradation/vietnam_pipeline.py',
]
for f in required:
    assert Path(f).exists(), f'MISSING: {f}'
    print(f'  ✓ {f}')
print(f'\n✓ Code ready at {DST}')

## 3. Copy sota_comparison.py vào project/scripts

Nếu code project đã có sẵn `scripts/sota_comparison.py` thì bỏ qua cell này. Nếu chưa, upload file `sota_comparison.py` lên Kaggle Dataset riêng rồi điền path bên dưới.

In [ ]:
from pathlib import Path
import shutil

PROJECT_DIR = Path('/kaggle/working/vn-old-portrait-restoration')
scripts_dir = PROJECT_DIR / 'scripts'

# Tìm sota_comparison.py trong input (nếu đã upload riêng) hoặc đã trong project
target = scripts_dir / 'sota_comparison.py'

if not target.exists():
    # Tìm trong /kaggle/input
    hits = list(Path('/kaggle/input').rglob('sota_comparison.py'))
    assert hits, ('Không tìm thấy sota_comparison.py. Upload file này lên một Kaggle Dataset '
                  'rồi Add Data vào notebook, hoặc bao gồm trực tiếp trong code project zip.')
    shutil.copy(hits[0], target)
    print(f'Copied {hits[0]} -> {target}')
else:
    print(f'✓ Đã có sẵn: {target}')

## 4. Install deps + patch GFPGAN basicsr

In [ ]:
!pip install -q einops lpips scikit-image
!pip install -q gfpgan==1.3.8 basicsr==1.4.2 facexlib==0.3.0 --force-reinstall --no-deps

import glob
candidates = glob.glob('/usr/local/lib/python*/dist-packages/basicsr/data/degradations.py')
assert candidates, 'basicsr không cài được'
degradations_file = candidates[0]

with open(degradations_file) as f:
    content = f.read()

old = 'from torchvision.transforms.functional_tensor import rgb_to_grayscale'
new = 'from torchvision.transforms.functional import rgb_to_grayscale'

if old in content:
    content = content.replace(old, new)
    with open(degradations_file, 'w') as f:
        f.write(content)
    print(f'✓ Patched {degradations_file}')
else:
    print('  (already patched)')

import torch
print(f'PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')
try:
    from gfpgan import GFPGANer
    print('✓ GFPGAN import OK')
except Exception as e:
    print(f'✗ GFPGAN fail: {e}')

## 5. RESTART KERNEL → chạy lại cells 1-4 → tiếp tục từ đây

## 6. Clone CodeFormer repo + download codeformer.pth

Bật Internet trong settings notebook (Settings → Internet → ON).

Repo CodeFormer ~50MB, weight `codeformer.pth` ~376MB.

In [ ]:
import os, subprocess
from pathlib import Path

CODEFORMER_REPO = '/kaggle/working/CodeFormer'
CODEFORMER_WEIGHTS_DIR = Path(CODEFORMER_REPO) / 'weights' / 'CodeFormer'
CODEFORMER_CKPT = CODEFORMER_WEIGHTS_DIR / 'codeformer.pth'

# Clone repo
if not Path(CODEFORMER_REPO).exists():
    print('Cloning CodeFormer repo...')
    subprocess.run(
        ['git', 'clone', '--depth=1',
         'https://github.com/sczhou/CodeFormer.git', CODEFORMER_REPO],
        check=True,
    )
    print(f'✓ Cloned to {CODEFORMER_REPO}')
else:
    print(f'✓ Repo đã có: {CODEFORMER_REPO}')

# Download codeformer.pth nếu chưa có
CODEFORMER_WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
if not CODEFORMER_CKPT.exists():
    print('Downloading codeformer.pth (~376 MB)...')
    url = 'https://github.com/sczhou/CodeFormer/releases/download/v0.1.0/codeformer.pth'
    subprocess.run(['wget', '-q', '--show-progress', url, '-O', str(CODEFORMER_CKPT)], check=True)
    print(f'✓ Downloaded {CODEFORMER_CKPT.stat().st_size/1e6:.1f} MB')
else:
    print(f'✓ Weight đã có: {CODEFORMER_CKPT}')

print(f'\nCODEFORMER_REPO = {CODEFORMER_REPO}')
print(f'CODEFORMER_CKPT = {CODEFORMER_CKPT}')

## 7. Find checkpoint paths

In [ ]:
from pathlib import Path
import torch

# Stage 3
stage3_candidates = [p for p in Path('/kaggle/input').glob('**/best.pth')
                       if p.stat().st_size >= 100_000_000]
STAGE3_CKPT = None
for p in stage3_candidates:
    try:
        ckpt = torch.load(p, map_location='cpu', weights_only=False)
        stage = ckpt.get('stage', None)
        fusion_keys = [k for k in ckpt.get('fusion', {}).keys() if 'blocks.' in k]
        block_indices = set()
        for k in fusion_keys:
            try:
                block_indices.add(int(k.split('blocks.')[1].split('.')[0]))
            except: pass
        num_blocks = max(block_indices) + 1 if block_indices else 0
        if stage == 3 or num_blocks >= 3:
            STAGE3_CKPT = str(p)
            print(f'  Stage 3: {p} ({num_blocks} fusion blocks)')
            break
    except Exception as e:
        print(f'  Skip {p}: {e}')
assert STAGE3_CKPT, 'Stage 3 ckpt not found'

# GFPGAN v1.4
GFPGAN_PRE = next(Path('/kaggle/input').rglob('GFPGANv1.4.pth'), None)
assert GFPGAN_PRE, 'GFPGANv1.4.pth not found'
GFPGAN_PRE = str(GFPGAN_PRE)

# CelebA-HQ
CELEBA = None
for p in Path('/kaggle/input').rglob('*'):
    if p.is_dir() and 'celeb' in p.name.lower():
        n_img = sum(1 for _ in p.rglob('*.jpg')) + sum(1 for _ in p.rglob('*.png'))
        if n_img > 1000:
            CELEBA = str(p)
            break
assert CELEBA, 'CelebA-HQ not found'

print(f'\nSTAGE3_CKPT      = {STAGE3_CKPT}')
print(f'GFPGAN_PRE       = {GFPGAN_PRE}')
print(f'CODEFORMER_CKPT  = {CODEFORMER_CKPT}')
print(f'CODEFORMER_REPO  = {CODEFORMER_REPO}')
print(f'CELEBA           = {CELEBA}')

## 8. Run SOTA comparison

~25-35 phút trên T4. Output ở `/kaggle/working/sota_results/`.

In [ ]:
# Clear cache import cũ để pick up sota_comparison.py mới copy
import sys
for mod in list(sys.modules.keys()):
    if mod.startswith('scripts'):
        del sys.modules[mod]

from scripts.sota_comparison import run_sota_comparison

results = run_sota_comparison(
    stage3_ckpt=STAGE3_CKPT,
    gfpgan_pretrained=GFPGAN_PRE,
    codeformer_ckpt=str(CODEFORMER_CKPT),
    codeformer_repo=CODEFORMER_REPO,
    test_dir=CELEBA,
    output_dir='/kaggle/working/sota_results',
    n_samples=200,
    levels=['medium', 'hard'],
    image_size=256,
    save_samples=10,
    codeformer_w=0.5,
    device='cuda',
    seed=42,
)

## 9. Verify output + xem LaTeX table

In [ ]:
from pathlib import Path
print('=== Output files ===')
for f in sorted(Path('/kaggle/working/sota_results').rglob('*')):
    if f.is_file():
        print(f'  {f.relative_to("/kaggle/working/sota_results")}  ({f.stat().st_size/1024:.1f} KB)')

print('\n=== LaTeX table (paste vào Ch5) ===')
print((Path('/kaggle/working/sota_results') / 'sota_table.tex').read_text())

## 10. View qualitative grid (5-column: LQ | GFPGAN | CodeFormer | Stage 3 | HQ)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

samples_dir = Path('/kaggle/working/sota_results/samples_sota')
samples = sorted(samples_dir.glob('*.png'))[:8]
assert samples, f'Không tìm thấy sample trong {samples_dir}'

fig, axes = plt.subplots(len(samples), 1, figsize=(16, 3.2 * len(samples)))
if len(samples) == 1:
    axes = [axes]

for ax, img_path in zip(axes, samples):
    img = Image.open(img_path)
    ax.imshow(img)
    ax.set_title(f'{img_path.stem}   |   LQ → GFPGAN v1.4 → CodeFormer → Stage 3 (ours) → HQ', fontsize=10)
    ax.axis('off')

plt.tight_layout()
out_path = '/kaggle/working/sota_results/qualitative_sota.png'
plt.savefig(out_path, dpi=80, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

## 11. Backup lên Kaggle Dataset (optional)

In [ ]:
import os, json
from pathlib import Path

os.environ['KAGGLE_USERNAME'] = 'mrtaintou'
os.environ['KAGGLE_KEY'] = 'PUT_YOUR_KEY_HERE'

OUT_DIR = Path('/kaggle/working/sota_results')

with open(OUT_DIR / 'dataset-metadata.json', 'w') as f:
    json.dump({
        'title': 'VN SOTA Comparison Results',
        'id': 'mrtaintou/vn-sota-comparison-results',
        'licenses': [{'name': 'CC0-1.0'}]
    }, f)

!kaggle datasets create -p /kaggle/working/sota_results --dir-mode zip